# M16 — Verified NFL Outcomes, Cross-Domain Generalization & Incremental Predictive Value (Colab laboratory)

**Status: authored, never executed by the author.** Every number this notebook shows comes from the tables it writes in your run directory. Gates start at `NOT_EVALUATED` and only change from computed tables.

## 00 — Scientific introduction

**Question.** Does 10 Hz Combine tracking carry generalizable, reproducible, athlete-specific information — and does it improve held-out prediction of NFL movement outcomes beyond the official Combine measurements (M1 = `draft_year, height, weight, forty, ten`)? A rigorous negative answer is an acceptable result.

**What M15 left us (observed, exploratory).**
- LSTM retrieval: Procrustes all-position top-1 ≈ 6.0 %, MRR ≈ 0.164, pair AUC ≈ 0.770; InfoNCE top-1 ≈ 3.8 %, MRR ≈ 0.149, AUC ≈ 0.790. Position-aware top-1 chance is ≈ 6 % in a typical fold, so the unrestricted gains may be position information.
- Harder pools: Procrustes same-position ΔMRR ≈ +0.010 and stopwatch-twin ΔMRR ≈ −0.005, InfoNCE same-position ΔMRR ≈ −0.009 — intervals cross zero. **Cross-domain identity beyond position / stopwatch is NOT established.**
- Domain AUC 0.97–0.98, LSTM effective rank 6–8 of 32, InfoNCE alignment harmed repeated-trial AUC in 3 / 5 folds, none of 80 spectral configurations passed measurement validity.
- E15.C (preservation) is blocked: `m15.preservation.robustness` reads `cfg.dropout_rates`, which `M15Config` does not define. M16 reports this as missing evidence; it does not work around it.
- Exploratory benchmark on `nfl_speed_q90_median` (a different target from the official `nfl.q90_shared`): position-only RMSE 0.4108 vs position + engineered tracking 0.4308 yd/s. Never compared with the official target.

**Four tracks.** A — verified NFL targets (registry, reliability, laboratory, context support) · B — controlled representation transfer (provenance, controlled retrieval, repeatability) · C — incremental NFL prediction (ladder B0–B7, controls, positions, drill hypotheses) · D — scientific gates and the report.

**Retrieval ≠ prediction.** Finding the same athlete across domains (G16-D) says nothing about predicting his NFL outcome (G16-C), and vice versa. The two gates are computed independently.

**Data dependencies (all read only).** M12 bundle · M12 / M13 / M14 experiments (persisted official OOF) · M14 run (athlete metrics + measurement status) · M15 run (sequence features, inductive embeddings, checkpoints, gate) · optional legacy exploratory M15 folders (provenance audit only).

**How to run.** Edit only the configuration cell (01). Then *Run all*. Each step is guarded: a failure or a block is recorded in `manifests/m16_status.json` and later independent steps still run. Re-running reloads every step whose saved input signature still matches (force a step with `force_steps`).

## 01 — Environment and package setup

In [ ]:
# Setup 1 — mount Drive and install nflbowl from a PINNED commit (the Colab PyTorch / CUDA build is never replaced)
from google.colab import drive
drive.mount('/content/drive')
GIT_COMMIT = 'main'   # replace by the commit SHA you want to reproduce, e.g. 'a5a2290'
REPO = f'git+https://github.com/mybenkhadda/NFL_bowl.git@{GIT_COMMIT}'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown
print('installed from', REPO, '— if nflbowl was already imported in this session: Runtime → Restart session, then continue from Setup 2')

In [ ]:
# Setup 2 — versions, hardware and imports
import gc, json, os, platform, sys
import importlib.metadata as im
import numpy as np
import polars as pl
from IPython.display import Markdown, display
import nflbowl
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7), 'nflbowl < 0.7.0 is imported: restart the session and rerun from Setup 2'
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m15 import plots as m15plots
from nflbowl.lab.m16 import config as m16cfg, m1_verification as m1v, pipeline as P, plots as PL
from nflbowl.lab.m16 import target_audit as TA, controlled_retrieval as CR, profiles as PR, reporting as REP

def install_record():
    rec = {'nflbowl': nflbowl.__version__, 'python': sys.version.split()[0], 'platform': platform.platform(), 'cpu_count': os.cpu_count()}
    try:
        rec['direct_url'] = json.loads(im.distribution('nflbowl').read_text('direct_url.json') or '{}')
    except Exception as e:
        rec['direct_url'] = f'unavailable: {e}'
    try:
        import torch
        rec.update(torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
    except ImportError:
        rec['torch'] = 'not installed (InfoNCE arms will be skipped and recorded)'
    for pkg in ('polars', 'numpy', 'matplotlib', 'scipy', 'lightgbm', 'catboost', 'umap-learn', 'ipywidgets', 'plotly'):
        try:
            rec[pkg] = im.version(pkg)
        except im.PackageNotFoundError:
            rec[pkg] = 'not installed'
    return rec

INSTALL = install_record()
print(json.dumps(INSTALL, indent=1, default=str))

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). M12 / M13 / M14 / M15 directories are read only; nothing is written there.
CFG = {
    'experiment_name': 'm16_verified_baselines_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'm15_run_dir': '/content/drive/MyDrive/NFLBowl/experimental_lab/m15_spectral_temporal_v1',
    'legacy_m15_dirs': [],                 # exploratory STFT / LSTM notebook folders (provenance audit only)
    'extra_oof_sources': [],               # further experiment folders holding evaluation/records/transfer_oof__*.parquet
    'exploratory_target_path': '',         # parquet / csv with nfl_id + nfl_speed_q90_median (kept separate, never compared)
    'reproduction_tol': 1e-8,
    'learners': ['enet', 'pooled'],        # + 'lightgbm', 'catboost' when installed (BLOCKED_MISSING_INPUT otherwise)
    'ladder_dim': 32,                      # pre-declared latent dimension of B3–B5 (never chosen on results)
    'n_shuffle': 20,                       # within-position shuffled-embedding refits per rung (negative control)
    'task_targets_in_ladder': 'eligible',  # 'all' = also unfrozen task targets (labelled)
    'force_steps': [],                     # e.g. ['E16.B5'] to recompute a step despite a valid cache
    'seed': 42,
}
cfg = m16cfg.resolve(CFG)
run = m16cfg.M16Run(cfg.run_dir, cfg)
ENV = {**handoff.environment(cfg.m13()), **INSTALL}
run.snapshot_config(cfg, ENV)
print('run dir            ', run.root)
print('config fingerprint ', cfg.fingerprint())
print('persisted OOF      ', [str(p) for p in cfg.experiment_dirs()] or 'none found (G16-B persisted criterion will be NOT_EVALUATED)')

def show(name, fam, n=30, cols=None):
    if run.has(name, fam):
        df = run.frame(name, fam)
        display(Markdown(f'**{fam}/{name}** — {df.height:,} rows')); display(df.select([c for c in cols if c in df.columns]) if cols else df.head(n))
    else:
        print(f'· {fam}/{name}: not produced in this run')

def fig(name, fn, *a, q='', c='', **k):
    try:
        f, d = fn(*a, **k)
    except Exception as e:  # a plotting error never stops the laboratory; it is printed and the step result stays valid
        print(f'figure {name} skipped: {type(e).__name__}: {e}')
        return
    run.figure(f, name, data=d, question=q, caption=c)

def step(sid, fn, note, requires=()):
    out = S.run_step(run, run.ledger, sid, fn, note=note, requires=list(requires))
    rec = run.ledger.records.get(sid, {})
    st, why = rec.get('status'), str(rec.get('reason', ''))[:300]
    print(f'[{sid}] {st} — {why}')
    return out

display(run.ledger.table())

In [ ]:
# Setup 4 — open the M16 experiment (refuses another bundle or science hash) and the shared data view
EXP = step('E16.open', lambda ctx: handoff.open_experiment(cfg.m13(), run), 'open the M16 experiment on the official bundle / science')
if EXP is None:
    display(run.ledger.table())
    raise SystemExit('the experiment could not be opened — see the reason above and logs/')
CFG15 = P.m15_config(cfg)
DATA16 = P.data_from_experiment(EXP, CFG15)
FEAT = P.feature_table(cfg, DATA16, CFG15, run)
print('experiment', EXP.id, '· bundle', EXP.bundle.fingerprint, '· science', EXP.cfg.science_hash())
print('sequences', DATA16.index.height, '· feature rows', FEAT.height)

## 02 — Data audit (E16.A9)

Two linkage definitions are kept side by side: the `linked` flag of `athletes.parquet` (documented 401) and athletes **observed in both domains** (documented 399). Retrieval uses observed-dual only; the official ladder keeps the flag. Fold integrity (athlete-disjoint, five folds, index = experiment fold map) is a hard check (FAILED_VALIDATION). Documented counts are compared, not asserted.

In [ ]:
COH = step('E16.A9', lambda ctx: P.run_cohort(cfg, EXP, DATA16, run, ctx), 'cohort, dual-domain reconciliation, fold integrity', ['E16.open'])
for name in ('documented_checks', 'dual_domain_summary', 'fold_integrity', 'funnel', 'exposure_by_position', 'artifact_inventory'):
    show(name, 'cohort', 60)
if run.has('dual_domain', 'cohort'):
    dd = run.frame('dual_domain', 'cohort')
    display(Markdown('**Athletes where the two linkage definitions disagree**')); display(dd.filter(~pl.col('reconciliation').str.starts_with('agree')))

In [ ]:
if run.has('funnel', 'cohort'):
    fig('M16_02_01_cohort_funnel', PL.cohort_funnel, run.frame('funnel', 'cohort'), q='How many athletes can each analysis use?', c='Athletes per cohort step, stacked by position.')
if run.has('fold_position', 'cohort'):
    fig('M16_02_02_fold_position', PL.heat, run.frame('fold_position', 'cohort'), 'position_group', 'fold', 'n_athletes', 'Athletes per fold × position (athlete-disjoint folds)',
        q='Are positions balanced across the five folds?', c='Athletes per outer fold and position.')
if COH is not None:
    fig('M16_02_03_exposure', PL.exposure_distribution, COH['exposure'], q='How unequal is NFL exposure?', c='NFL windows per athlete by position (log).')
if run.has('composition_kind', 'cohort'):
    ck = run.frame('composition_kind', 'cohort').with_columns(pl.concat_str([pl.col('domain'), pl.col('kind')], separator=' · ').alias('domain_kind'))
    fig('M16_02_04_composition', PL.heat, ck, 'domain_kind', 'position_group', 'n_athletes', 'Athletes per drill / NFL demand kind × position', annotate='n_windows',
        q='Which movement contexts exist for which positions?', c='Athletes (and windows) per domain × kind × position.')

## 03 — Official M1 verification (E16.A0–A1)

Recompute the official ladder M1 and compare, athlete by athlete at 1e-8, with its independent refit and every compatible persisted official OOF record. A mismatch is FAILED_VALIDATION **after** all evidence is written; it suspends every comparison against M1 (G16-B hard stop). A new model with other settings is never called a reproduction.

In [ ]:
def step_m1(ctx):
    v = m1v.verify_m1(cfg, run, EXP, ctx)
    ctx['outputs'] += ['tables/baselines/m1_reproducibility', 'manifests/m1_verification.json']
    return v

M1 = step('E16.A1', step_m1, 'official M1 contract + reproducibility', ['E16.open'])
sp = run.root / 'manifests' / 'm1_verification.json'
SUMMARY = json.loads(sp.read_text()) if sp.is_file() else None
if SUMMARY:
    print('VERDICT:', SUMMARY['verdict'], '| internal ok:', SUMMARY['internal_ok'], '| checks ok:', SUMMARY['checks_ok'], '| compatible persisted:', SUMMARY['n_compatible_records'])
    print('failures:', SUMMARY['failures'] or 'none')
for name in ('m1_contract_columns', 'm1_reproducibility', 'm1_checks', 'm1_persisted_records', 'm1_metrics'):
    show(name, 'baselines')
if run.has('m1_athlete_diffs', 'baselines'):
    d = run.frame('m1_athlete_diffs', 'baselines')
    display(Markdown('**mismatched athletes**')); display(d.filter(pl.col('mismatch')).head(50))
    fig('M16_03_01_m1_reproduction_diffs', PL.reproduction_diffs, d, cfg.reproduction_tol, q='Is the official M1 reproduced athlete by athlete?',
        c='|Δ prediction| per athlete and comparison (log; exact zeros at 1e-17); dashed = tolerance.')
if run.has('m1_metrics', 'baselines'):
    fig('M16_03_02_rmse_by_fold', PL.metrics_by_fold, run.frame('m1_metrics', 'baselines'), q='Do recomputed and persisted baselines agree fold by fold?',
        c='Out-of-fold RMSE by fold on nfl.q90_shared (descriptive; equality is judged on predictions).')
T0X = step('E16.A0-x', lambda ctx: m1v.exploratory_target_audit(cfg, EXP.bundle.athletes, run), 'exploratory target audit (descriptive, never compared)', ['E16.open'])
if T0X is not None:
    print({k: T0X[k] for k in ('overlap_athletes', 'only_T0', 'only_T0x', 'note')})

## 04 — NFL target laboratory (E16.A2, A3, A7, A8)

Targets from NFL data only: **T0** official `nfl.q90_shared` (primary) · **T1** high-speed capacity · **T2 / T2b** acceleration at matched entry speed · **T3** braking (complete events) · **T4** reacceleration (censored propulsion excluded, share reported) · **T5** redirection speed retention (heading-rate turns; not geometric curvature) · **T6** consistency. These are constructed movement outcomes, not football performance.

Then reliability and the one-time freeze (E16.A3), the laboratory (E16.A7: distributions, variance structure, exposure / duration / context / draft-year relations, threshold / aggregation sensitivity, subsample stability, correlations, selection, censoring, scorecard) and the Combine-vs-NFL context common support (E16.A8: blocked when overlap is insufficient — never extrapolated).

In [ ]:
T = step('E16.A2', lambda ctx: P.run_targets(cfg, DATA16, FEAT, CFG15, run, ctx), 'NFL task targets', ['E16.open'])
def step_a3(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return P.run_reliability(cfg, DATA16, T, run, ctx)
REL = step('E16.A3', step_a3, 'target reliability + immutable freeze', ['E16.A2'])
show('definitions', 'target', cols=['target_id', 'column', 'unit', 'role', 'aggregation'])
show('coverage', 'target', 80)
for name in ('reliability', 'exposure_dependence', 'context_share', 'gate_a'):
    show(name, 'target')
if REL is not None:
    print('freeze:', {k: REL['freeze'].get(k) for k in ('primary', 'eligible_task_targets', 'conflict', 'written_at')})

In [ ]:
def step_a7(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return P.run_target_lab(cfg, DATA16, T, REL, EXP.bundle.athletes, run, ctx)
LAB = step('E16.A7', step_a7, 'target laboratory', ['E16.A2'])
for name in ('scorecard', 'measurement_reliability', 'variance', 'relations', 'min_obs_sensitivity', 'aggregation_sensitivity', 'subsample_stability',
             'selection_bias', 'censoring'):
    show(name, 'exploration', 60)

In [ ]:
if run.has('coverage', 'target'):
    fig('M16_04_01_target_coverage', PL.target_coverage, run.frame('coverage', 'target'), q='Which NFL targets are observable, for how many athletes?',
        c='Athletes per target and position after the minimum-exposure rules.')
if LAB is not None:
    LONG = LAB['long']
    fig('M16_04_02_target_violins', PL.target_violins, LONG, q='How do targets differ within and between positions?', c='One value per athlete; n under each violin.')
    for tid in LONG['target_id'].unique().sort().to_list():
        k = TA.kde_grid(LONG, tid)
        if k.height:
            fig(f'M16_04_03_kde_{tid}', PL.target_kde, k, LONG.filter(pl.col('target_id') == tid)['unit'][0], q=f'What does {tid} look like?', c='Gaussian KDE (Scott bandwidth).')
    fig('M16_04_04_variance', PL.variance_bars, LAB['variance'], q='How much target variance is within position?', c='η² with athlete-bootstrap CI.')
    fig('M16_04_05_threshold', PL.lines_by_group, LAB['min_obs_sensitivity'], 'threshold', 'spearman_vs_frozen', 'target_id',
        'Target stability vs the minimum-exposure rule (Spearman with the frozen values)', 'minimum windows / events per athlete', 'Spearman vs frozen target', ref=0.8,
        q='Do the athlete rankings depend on the exposure threshold?', c='Each target rebuilt at every threshold; dashed 0.8.')
    fig('M16_04_06_subsample', PL.lines_by_group, LAB['subsample_stability'], 'fraction', 'spearman_mean', 'target_id',
        'Target stability vs sample size (random fraction of each athlete’s observations)', 'fraction of observations kept', 'Spearman with the full-data value',
        lo='spearman_lo', hi='spearman_hi', q='How stable is a target with less NFL data?', c='Mean and 2.5–97.5 % over replicates.')
    corr = LAB['correlations'].filter(pl.col('target_a') != pl.col('target_b'))
    fig('M16_04_07_target_correlations', PL.heat, corr, 'target_a', 'target_b', 'spearman_within_position', 'Within-position Spearman between targets (n = common cohort)',
        fmt='{:.2f}', cmap='RdBu_r', vmin=-1, vmax=1, annotate='n_common', q='Are the targets distinct outcomes?', c='Position-centred Spearman; n per cell.')
    r = LAB['relations'].filter(pl.col('lo').is_not_null())
    fig('M16_04_08_relations', PL.delta_forest, r, ['target_id', 'covariate'], 'spearman_within_position', 'lo', 'hi', 'within-position Spearman (95 % athlete bootstrap)',
        'Do targets depend on exposure, duration, demand mix or draft year?', color_by='covariate', q='What besides the athlete drives each target?', c='|ρ| > 0.3 is flagged.')
if REL is not None:
    fig('M16_04_09_reliability_vs_exposure', PL.reliability_vs_exposure, REL['curve'], q='How much NFL exposure makes a target reliable?',
        c='Split-half reliability (Spearman–Brown, position-residualised) vs observations per half; dashed 0.7.')

In [ ]:
# Optional interactive explorer (ipywidgets; the static figures above are the record)
try:
    import ipywidgets as W
    if LAB is not None:
        dd = W.Dropdown(options=sorted(LAB['long']['target_id'].unique().to_list()), description='target')
        def _show(tid):
            display(LAB['distributions'].filter(pl.col('target_id') == tid))
            display(LAB['scorecard'].filter(pl.col('target_id') == tid))
        W.interact(_show, tid=dd)
except ImportError:
    print('ipywidgets not installed — static outputs only')

In [ ]:
def step_a8(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return P.run_context(cfg, DATA16, FEAT, T, run, ctx)
CTX = step('E16.A8', step_a8, 'Combine-drill vs NFL common support + context-matched outcomes', ['E16.A2'])
for name in ('support', 'overlap', 'matched_summary', 'matched_vs_global'):
    show(name, 'context', 60)
if run.has('overlap', 'context'):
    fig('M16_04_10_context_overlap', PL.overlap_profile, run.frame('overlap', 'context'), q='Do Combine drills resemble NFL movement contexts?',
        c='5–95 % ranges and medians per covariate; OVL = shared histogram mass.')
if run.has('support', 'context'):
    sup = run.frame('support', 'context').with_columns(pl.lit('common support').alias('col'))
    fig('M16_04_11_support_status', PL.status_matrix, sup, 'drill', 'col', 'status', 'Context common support per drill (blocked = no extrapolated matching)',
        q='Which drill-to-NFL comparisons are allowed?', c='COMPLETED or BLOCKED_INSUFFICIENT_OVERLAP per drill.')

## 05 — Embedding and checkpoint registry (E16.B4)

Every M15 embedding / checkpoint is audited from its files: fold, training cohort, held-out exclusion, bundle fingerprint, channel order and scaling, input transform, checksum, export schema. States: `VERIFIED_FOLD_SAFE` · `EXPLORATORY_ONLY` · `UNKNOWN_PROVENANCE` · `INVALID_LEAKAGE`. **Only VERIFIED_FOLD_SAFE representations enter supervised comparisons**; everything else is blocked, never silently accepted. The G15 decision and the E15.C preservation evidence (including the `dropout_rates` blocker) are attached.

In [ ]:
PROV = step('E16.B4', lambda ctx: P.run_provenance(cfg, EXP, DATA16, run, ctx), 'M15 representation provenance registry', ['E16.open'])
for name in ('registry', 'registry_counts', 'registry_folds', 'legacy'):
    show(name, 'representation', 60)
pe = run.root / 'manifests' / 'm15_preservation_evidence.json'
if pe.is_file():
    print('E15.C preservation evidence:', pe.read_text())
if run.has('registry_folds', 'representation'):
    rf = run.frame('registry_folds', 'representation').with_columns(pl.concat_str([pl.col('arch'), pl.lit('_d'), pl.col('d').cast(pl.String), pl.lit('_s'), pl.col('seed').cast(pl.String)]).alias('rep'),
                                                                    pl.concat_str([pl.lit('fold '), pl.col('fold').cast(pl.String)]).alias('f'))
    fig('M16_05_01_provenance', PL.status_matrix, rf, 'rep', 'f', 'status', 'Provenance per representation × fold',
        q='Which M15 artifacts may enter supervised comparisons?', c='Four provenance states; reasons in tables/representation/registry_folds.')
REG = PROV['registry'] if PROV is not None else (run.frame('registry', 'representation') if run.has('registry', 'representation') else None)

## 06 — Controlled retrieval and repeatability (E16.B1, B3, B5, B6)

E16.B1 / B3 keep the session-4 confirmatory retrieval of G15-eligible representations and the leakage audit. E16.B5 scores **every verified representation**, the STFT ⊕ LSTM fusion and the controls (F0, train-standardised kinematics, analytic position-only, random) on identical folds and pools: all · same position · stopwatch twins · physical-profile twins, per drill scope. Chance is pool-specific; the decisive test is each representation vs the **strongest control on the same queries**. E16.B6 asks whether alignment damages repeatability.

In [ ]:
RET = step('E16.B1', lambda ctx: P.run_retrieval(cfg, EXP, DATA16, FEAT, run, ctx), 'confirmatory retrieval of G15-eligible representations (session 4)', ['E16.open'])
def step_b3(ctx):
    if RET is None or not RET['reps']:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no G15-eligible representation to audit', missing=['E16.B1'])
    return P.run_leakage(cfg, EXP, RET['reps'], run, ctx)
LEAK = step('E16.B3', step_b3, 'leakage audit (combine_only_inductive / diagnostic_only)', ['E16.B1'])
def step_b5(ctx):
    if REG is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no provenance registry (run E16.B4)', missing=['E16.B4'])
    return P.run_controlled_retrieval(cfg, EXP, DATA16, FEAT, REG, run, ctx)
CRV = step('E16.B5', step_b5, 'controlled retrieval vs controls', ['E16.open'])
gc.collect()
show('controlled_summary', 'retrieval', 80)
if run.has('controlled_vs_controls', 'retrieval'):
    vc = run.frame('controlled_vs_controls', 'retrieval')
    display(Markdown('**vs the strongest control (Holm across representations × arms per pool / scope)**')); display(vc.filter(pl.col('strongest')))
for name in ('controlled_errors', 'controlled_notes'):
    show(name, 'retrieval', 40)

In [ ]:
if run.has('controlled_summary', 'retrieval'):
    cs = run.frame('controlled_summary', 'retrieval')
    for scope in cfg.retrieval_scopes:
        fig(f'M16_06_01_retrieval_{scope}', PL.retrieval_vs_chance, cs, scope, 'unaligned', q='Does any representation beat pool-specific chance?',
            c=f'MRR with athlete-bootstrap CI per pool, scope {scope}; | = chance of that pool.')
if run.has('controlled_vs_controls', 'retrieval'):
    vc = run.frame('controlled_vs_controls', 'retrieval').filter(pl.col('strongest') & (pl.col('scope') == 'all')).sort('pool', 'rep', 'arm')
    fig('M16_06_02_vs_strongest_control', PL.delta_forest, vc, ['pool', 'rep', 'arm', 'control'], 'delta_mrr', 'lo', 'hi', 'ΔMRR vs the strongest control (same queries)',
        'Representation minus strongest control — the decisive retrieval comparison', color_by='pool', q='Is there similarity beyond position and stopwatch?',
        c='Paired athlete bootstrap 95 % CI; > 0 = better than every control.')
if CRV is not None:
    SC = CRV['scores']
    reps = [r for r in SC['rep'].unique().sort().to_list() if r not in ('random',)]
    for pool in ('same_position', 'twins'):
        fig(f'M16_06_03_rank_hist_{pool}', PL.rank_histogram, SC, reps, pool, q='Where does the true NFL match rank?', c='Relative rank density; 1 = chance.')
    if CRV.get('pool_meta') is not None and CRV['pool_meta'].height:
        fig('M16_06_04_pool_sizes', PL.pool_sizes, CRV['pool_meta'], q='How large are the candidate pools?', c='Candidates per query and pool.')
    if CRV.get('sims') is not None and CRV['sims'].height:
        for r in [x for x in reps if x not in ('F0', 'kinematic')]:
            fig(f'M16_06_05_similarity_{r}', PL.similarity_distributions, CRV['sims'], r, q='Are true pairs more similar than impostors?', c='Same-position pool, held-out athletes.')
    for r in reps[:6]:
        display(Markdown(f'**Gallery — {r} (same position, unaligned)**')); display(CR.gallery(SC, r))

In [ ]:
def step_b6(ctx):
    if REG is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no provenance registry (run E16.B4)', missing=['E16.B4'])
    return P.run_repeatability(cfg, EXP, DATA16, REG, run, ctx)
RPT = step('E16.B6', step_b6, 'alignment damage, geometry, ICC, probes, M1 re-encoding, duration', ['E16.open'])
for name in ('repeat_damage', 'representation_geometry', 'representation_m1_relation'):
    show(name, 'retrieval', 60)
if RPT is not None and RPT['repeat_rows'].height:
    for r in RPT['repeat_rows']['rep'].unique().to_list():
        fig(f'M16_06_06_alignment_{r}', PL.alignment_slopes, RPT['repeat_rows'], r, q='Does alignment damage repeatability?', c='Repeat AUC per held-out fold; dotted 0.5.')
    if RPT.get('representation_m1_relation') is not None:
        mr = RPT['representation_m1_relation'].group_by('rep', 'm1_column').agg(pl.col('ridge_r2_heldout').mean())
        fig('M16_06_07_m1_reencoding', PL.heat, mr, 'rep', 'm1_column', 'ridge_r2_heldout', 'Held-out R² of each official measurement from the representation',
            fmt='{:.2f}', cmap='Oranges', vmin=0, vmax=1, q='How much of the stopwatch / body size does the latent space re-encode?', c='Ridge fitted on training athletes, mean over folds.')

In [ ]:
# Descriptive latent views of one verified representation (fold 0): projection by domain / position and a same-position similarity matrix
from nflbowl.lab.m15 import latent as LT
REPS = P.chosen_reps(cfg, REG) if REG is not None else []
for rep in REPS:
    p = P._emb_path(cfg, rep, 0)
    if not p.is_file():
        continue
    emb = pl.read_parquet(p)
    name = rep['rep']
    try:
        proj = LT.projection(emb, {'arch': rep['arch'], 'd': rep['d'], 'fold': 0}, seed=cfg.seed)
        for by in ('domain', 'position_group'):
            fig(f'M16_06_08_projection_{name}_{by}', m15plots.projection_scatter, proj, by, q='How is the latent space organised?', c='Descriptive; not evidence of predictive value.')
    except Exception as e:
        print('projection skipped:', e)
    ids, Smat = CR.similarity_matrix(emb, None, 'WR')
    if ids.size >= 5:
        fig(f'M16_06_09_similarity_matrix_{name}', PL.similarity_heatmap, ids, Smat, f'{name} fold 0 WR',
            q='Do true pairs stand out within a position?', c='Cosine similarity, Combine vs NFL athlete means; diagonal = true pairs.')
    del emb
    gc.collect()

## 07 — Combine-only athlete features (E16.C0)

Per outer fold, features come from the fold-f encoder (never trained on fold-f athletes) and **Combine attempts only** — no NFL-derived representation is ever an input. Primary aggregation = mean (official convention); median, trimmed mean and ICC-weighted (weights from training athletes only) are sensitivity analyses; drill scopes (forty, agility, three-cone, short shuttle) feed the hypotheses. Missingness stays missing (indicators are fitted on training rows).

In [ ]:
def step_c0(ctx):
    if T is None or REG is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'needs targets (E16.A2) and the provenance registry (E16.B4)', missing=['E16.A2', 'E16.B4'])
    return P.run_features(cfg, EXP, FEAT, T['targets'], CTX['matched'] if CTX else None, REG, run, ctx)
FEATS = step('E16.C0', step_c0, 'fold-specific Combine-only athlete features + analysis frame', ['E16.A2', 'E16.B4'])
show('rung_plan', 'ladder')
for name in ('inventory', 'coverage', 'm1_relation', 'notes'):
    show(name, 'features', 60)
if FEATS is not None:
    for r, tb in FEATS['tables'].items():
        if 0 in tb and tb[0].width > 2:
            M = tb[0].drop('nfl_id').to_numpy()
            C = np.corrcoef(M[np.isfinite(M).all(1)].T)
            cols = [c for c in tb[0].columns if c != 'nfl_id']
            cdf = pl.DataFrame([{'a': cols[i], 'b': cols[j], 'r': float(C[i, j])} for i in range(len(cols)) for j in range(len(cols))])
            fig(f'M16_07_01_feature_corr_{r}', PL.heat, cdf, 'a', 'b', 'r', f'{r}: correlation of athlete-level latent features (fold 0)', fmt='{:.1f}', cmap='RdBu_r', vmin=-1, vmax=1,
                q='How redundant are the latent dimensions?', c='Pearson r between athlete-mean dimensions (fold-0 encoder).')

## 08 — Supervised baseline ladder (E16.A4, A6, C4)

E16.A4 is the session-4 baseline ladder that decides **G16-C** (POS · M1 · M1+F0 · M1+simple · M1+M14_reliable). E16.C4 is the laboratory ladder **B0–B7**: POS · M1 · M1+simple · M1+PCA · M1+STFT · M1+LSTM · M1+STFT+LSTM · M1+M14_reliable, on the official engine (train-fold design, inner-CV α, unpenalised position), the same athletes for every model, paired fold-stratified bootstrap, Holm on the pre-declared screen family. Representations without G15 GO are fitted but labelled `exploratory_screen`. B7 is blocked when no M14 metric passed — criteria are never relaxed. E16.A6 gives the minimum detectable effect.

In [ ]:
def step_a4(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return P.run_baselines(cfg, EXP, FEAT, T['targets'], REL['freeze'] if REL else None, run, ctx)
BASE = step('E16.A4', step_a4, 'baseline ladder for G16-C (session 4)', ['E16.A2', 'E16.A1'])
def step_a6(ctx):
    if BASE is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'analysis frame not available (run E16.A4)', missing=['E16.A4'])
    return P.run_detectability(cfg, BASE['data'], P.ladder_targets(cfg, REL['freeze'] if REL else None), run, ctx)
MDE = step('E16.A6', step_a6, 'planted-signal power / MDE', ['E16.A4'])
def step_c4(ctx):
    if FEATS is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'features not available (run E16.C0)', missing=['E16.C0'])
    return P.run_ladder16(cfg, EXP, FEATS, REL['freeze'] if REL else None, run, ctx)
LAD = step('E16.C4', step_c4, 'laboratory ladder B0–B7 (+ learners, sensitivity)', ['E16.C0'])
gc.collect()
show('status', 'baselines')
show('mde', 'detectability')
for name in ('rung_plan', 'metrics', 'paired', 'status', 'cohort_audit', 'improvement', 'paired_spearman', 'sensitivity_paired', 'notes'):
    show(name, 'ladder', 60)

In [ ]:
if BASE is not None and BASE['paired'].height:
    fig('M16_08_01_baseline_forest', PL.ladder_forest, BASE['paired'], 'M1', q='Does anything beat the official M1 (G16-C ladder)?',
        c='ΔRMSE vs M1 (> 0 = better), 95 % fold-stratified athlete bootstrap; grey = ±2 % margin.')
if MDE is not None:
    fig('M16_08_02_power', PL.power_curves, MDE['power'], q='What effect could this cohort detect?', c='Power vs planted variance share; dashed 0.8.')
if run.has('paired', 'ladder'):
    fig('M16_08_03_ladder_forest', PL.ladder_forest, run.frame('paired', 'ladder'), 'M1', q='Does any rung B2–B7 beat M1?',
        c='Laboratory ladder; identical athletes; grey = ±2 % practical margin.')
if run.has('intervals', 'ladder'):
    iv = run.frame('intervals', 'ladder')
    for tid in iv['target_id'].unique().sort().to_list():
        fig(f'M16_08_04_rmse_{tid}', PL.rmse_dots, iv, tid, q='How large are the errors, with uncertainty?', c='RMSE with fold-stratified athlete-bootstrap CI; axis from 0.')
if run.has('ranking', 'ladder'):
    rk = run.frame('ranking', 'ladder')
    for tid in rk['target_id'].unique().sort().to_list():
        fig(f'M16_08_05_within_position_{tid}', PL.within_position_ranking, rk.filter(pl.col('target_id') == tid), q='Does any rung rank players better within a position?',
            c='Within-position Spearman with 95 % athlete-bootstrap CI.')
if run.has('sensitivity_paired', 'ladder'):
    sp_ = run.frame('sensitivity_paired', 'ladder')
    fig('M16_08_06_sensitivity', PL.delta_forest, sp_, ['model'], 'delta_rmse', 'delta_lo', 'delta_hi', 'ΔRMSE vs M1 (target T0)', 'Aggregation and drill-scope sensitivity of the embedding rungs',
        q='Does the conclusion depend on how attempts are aggregated?', c='Each variant vs M1 on identical athletes.')

## 09 — Advanced evaluation (E16.C5, C6, C7 and descriptive scouting views)

Negative controls: embeddings **shuffled within position** (athlete link broken, position kept), random embeddings, reduced dimensions. A rung whose gain is inside the shuffled cloud is **not** athlete-specific. Position investigation: global vs separate vs partially pooled models (TE is small: read intervals, not points). Drill hypotheses E16-C1 (forty → high speed), C2 (short shuttle → braking / reacceleration), C3 (three-cone → redirection, context-matched) run only when sample size, validity, censoring and overlap allow.

In [ ]:
def need_ladder():
    if FEATS is None or LAD is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'needs E16.C0 and E16.C4', missing=['E16.C0', 'E16.C4'])
def step_c5(ctx):
    need_ladder(); return P.run_controls16(cfg, EXP, FEATS, LAD, run, ctx)
def step_c6(ctx):
    need_ladder(); return P.run_positions16(cfg, EXP, FEATS, LAD, run, ctx)
def step_c7(ctx):
    if FEATS is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'needs E16.C0', missing=['E16.C0'])
    ga = REL['gate_a'] if REL else (run.frame('gate_a', 'target') if run.has('gate_a', 'target') else None)
    sup = run.frame('support', 'context') if run.has('support', 'context') else None
    return P.run_hypotheses(cfg, EXP, FEATS, T, ga, sup, run, ctx)
CTRL = step('E16.C5', step_c5, 'negative controls (shuffled within position, random, reduced)', ['E16.C4'])
POSN = step('E16.C6', step_c6, 'position-specific models', ['E16.C4'])
HYP = step('E16.C7', step_c7, 'drill → NFL hypotheses E16-C1–C3', ['E16.C0'])
gc.collect()
show('summary', 'controls', 40)
for name in ('sample_sizes', 'eligibility', 'strategies', 'effects'):
    show(name, 'positions', 60)
for name in ('hypotheses_status', 'hypotheses_paired', 'hypotheses_declared'):
    show(name, 'hypotheses', 60)

In [ ]:
from nflbowl.lab import transfer_fairness as TF
if CTRL is not None and CTRL.get('null') is not None:
    for tid in CTRL['summary']['target_id'].unique().to_list():
        fig(f'M16_09_01_controls_{tid}', PL.null_strip, CTRL['null'], CTRL['summary'], tid, q='Is the gain athlete-specific?', c='Null clouds vs the observed ΔRMSE (diamond).')
if run.has('calibration_bins', 'ladder'):
    cb = run.frame('calibration_bins', 'ladder')
    for tid in cb['target_id'].unique().sort().to_list():
        ms = [m for m in ('POS', 'M1', 'M1+simple', 'M1+PCA', 'M1+STFT', 'M1+LSTM') if cb.filter((pl.col('target_id') == tid) & (pl.col('model') == m)).height]
        fig(f'M16_09_02_calibration_{tid}', PL.calibration_plot, cb.filter(pl.col('target_id') == tid), ms, tid, q='Are predictions calibrated?', c='Decile means; dashed = perfect.')
if run.has('stratified_errors', 'ladder'):
    se_ = run.frame('stratified_errors', 'ladder')
    for (tid, var), g in se_.group_by('target_id', 'variable'):
        ms = [m for m in ('POS', 'M1', 'M1+simple', 'M1+PCA', 'M1+LSTM') if g.filter(pl.col('model') == m).height]
        vname = var.replace(' ', '_')
        fig(f'M16_09_03_strata_{tid}_{vname}', PL.stratified_bars, g, ms, f'{tid} — RMSE by {var}', q='Where do models fail?', c='Strata partition the same cohort.')
if LAD is not None:
    for tid, w in LAD['wides'].items():
        ms = [c.removeprefix('pred_') for c in w.columns if c.startswith('pred_') and c.removeprefix('pred_') in ('POS', 'M1', 'M1+simple', 'M1+PCA', 'M1+STFT', 'M1+LSTM', 'M1+STFT+LSTM', 'M1+M14_reliable')]
        ra = TF.rank_agreement(w, ms)
        fig(f'M16_09_04_rank_agreement_{tid}', PL.heat, ra, 'model_a', 'model_b', 'spearman', f'{tid} — Spearman between model predictions', fmt='{:.2f}', cmap='Greens', vmin=0, vmax=1,
            q='Do augmented models re-rank athletes?', c='Agreement of OOF predictions (1 = identical ranking).')
if run.has('effects', 'positions'):
    ef = run.frame('effects', 'positions')
    cols = [c for c in ('delta_rmse', 'delta_lo', 'delta_hi') if c in ef.columns]
    if len(cols) == 3:
        fig('M16_09_05_position_effects', PL.delta_forest, ef.filter(pl.col('delta_lo').is_not_null()), ['target_id', 'position_group', 'model'], 'delta_rmse', 'delta_lo', 'delta_hi',
            'ΔRMSE vs M1 inside the position', 'Position-specific effects (small groups are exploratory)', color_by='position_group', q='Does any position benefit?',
            c='Paired athlete bootstrap within position; see sample_sizes for TE.')
if run.has('hypotheses_paired', 'hypotheses'):
    hp = run.frame('hypotheses_paired', 'hypotheses').filter(pl.col('reference') == 'M1')
    fig('M16_09_06_hypotheses', PL.delta_forest, hp, ['hypothesis', 'target_id', 'model'], 'delta_rmse', 'delta_lo', 'delta_hi', 'ΔRMSE vs M1',
        'Drill-to-NFL hypotheses — research hypotheses, not established transfer', color_by='hypothesis', q='Does a drill predict its matched NFL skill?', c='Holm within each hypothesis.')

In [ ]:
# Scouting-oriented DESCRIPTIVE views — no recommendation is built from a model that has not passed its gates
if T is not None:
    tcol = 'nfl.q90_shared' if 'nfl.q90_shared' in T['targets'].columns else None
    tg = T['targets'] if tcol else None
    if tg is None and EXP.bundle.athletes is not None and 'nfl.q90_shared' in EXP.bundle.athletes.columns:
        tg, tcol = EXP.bundle.athletes, 'nfl.q90_shared'
    for pos in ('WR', 'DB'):
        if tg is None:
            break
        prof = PR.by_outcome_quartile(DATA16, FEAT, tg, tcol, pos, seed=cfg.seed)
        if prof.height:
            fig(f'M16_09_07_forty_by_outcome_{pos}', PL.speed_profiles, prof, f'{pos}: forty speed curves by quartile of {tcol}', q='Does the forty shape differ by later NFL speed?',
                c='Mean and IQR per quartile of the NFL target within position.')
        curves, pairs = PR.stopwatch_twins(DATA16, FEAT, tg, EXP.bundle.athletes, tcol, pos, cfg.twin_forty_s)
        if curves.height:
            display(Markdown(f'**{pos}: stopwatch twins with the most different NFL outcome (pseudonymised)**')); display(pairs)
            fig(f'M16_09_08_stopwatch_twins_{pos}', PL.speed_profiles, curves, f'{pos}: same forty time, different {tcol}', q='Same stopwatch, same movement?',
                c='Mean forty speed curve per athlete; pairs with |Δforty| ≤ 0.03 s.')

## 10 — Scientific decision gates

`m16_gate` (G16 A–D, gates document; read by M17 / M18) and `m16_gate_extended` (one row per gate × subject with evidence, rule, observed value, CI, artifact, explanation and next action; states PASS · FAIL · EXPLORATORY · BLOCKED · NOT_EVALUATED). Document IDs are kept: **G16-C = incremental information (prediction)**, **G16-D = cross-domain similarity (representation transfer)**. No gate passes because its code exists; a nominally positive point estimate is not a PASS.

In [ ]:
G16 = step('G16', lambda ctx: P.run_gate(run, ctx), 'G16 A–D (gates document; read by M17 / M18)')
G16X = step('G16x', lambda ctx: P.run_gate_extended(cfg, run, ctx), 'G16 extended evidence table')
if G16 is not None:
    display(G16)
    fig('M16_10_01_gate_matrix', m15plots.gate_matrix, G16.with_columns(pl.lit('M16').alias('family')), 'G16', q='Are targets, baseline, increment and similarity verified?', c='G16 A–D.')
if G16X is not None:
    display(G16X.filter(pl.col('criterion').str.contains('overall')))
    display(G16X.filter(~pl.col('criterion').str.contains('overall')).select('gate', 'subject', 'status', 'observed', 'ci_lo', 'ci_hi', 'explanation', 'next_action'))
    fig('M16_10_02_gate_extended', PL.gate_extended_matrix, G16X, q='Which subjects pass which gate?', c='One row per gate × subject; state in each cell.')

## 11 — Final research report and hand-off

The report is generated from the saved tables only: what was evaluated, what passed, what failed, what remains uncertain, readiness for M17 and the pivots implied by the gates document.

In [ ]:
RPTX = step('E16.report', lambda ctx: P.run_report(cfg, run, ctx), 'evidence-based research report')
if RPTX:
    display(Markdown(RPTX))
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm16_handoff')
display(run.ledger.table())
for rel in ('manifests/m16_status.json', 'manifests/m1_verification.json', 'manifests/target_freeze_m16.json', 'manifests/target_definitions_m16.json',
            'manifests/dual_domain.json', 'manifests/m15_preservation_evidence.json', 'models/ladder_models.json', 'predictions/m16_ladder_oof_long.parquet',
            'retrieval/controlled_scores.parquet', 'tables/synthesis/m16_gate_extended.parquet', 'reports/markdown/m16_research_report.md'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print()
print('SEND BACK: manifests/stages.json, manifests/m16_status.json, logs/*_traceback.txt, manifests/m1_verification.json, manifests/m15_preservation_evidence.json,')
print('           tables/cohort/documented_checks.csv, tables/cohort/fold_integrity.csv, tables/target/gate_a.csv, tables/exploration/scorecard.csv,')
print('           tables/context/support.csv, tables/representation/registry.csv, tables/retrieval/controlled_vs_controls.csv, tables/retrieval/repeat_damage.csv,')
print('           tables/ladder/rung_plan.csv, tables/ladder/paired.csv, tables/ladder/status.csv, tables/ladder/cohort_audit.csv, tables/controls/summary.csv,')
print('           tables/hypotheses/hypotheses_status.csv, tables/synthesis/m16_gate.csv, tables/synthesis/m16_gate_extended.csv, reports/markdown/m16_research_report.md')